# 02 - Phase 1 Screening (4 models x 16 covariates x 2 windows = 128 experiments)

Tujuan: screening single-covariate untuk mencari covariate mana yang memberi improvement signifikan terhadap baseline (target-only) di 4 model tree-based.

## Desain
- **Models (4):** RandomForest, ExtraTrees, XGBoost, LightGBM
- **Covariates (16):** None (baseline) + 15 single covariates
- **Windows (2):** 30, 120
- **CV:** 5-fold expanding window (sama dengan modelling lama)
- **Hyperparameters:** default konservatif dari spec (bukan arbitrary). Tuning full ditunda ke Phase 2 untuk config terbaik.
- **Metric utama:** MAPE di level harga IHSG (bukan log-diff)

## Output
- `phase1_screening_results.csv` - 128 rows, semua metrik per experiment
- `phase1_screening_analysis.csv` - pass/fail per covariate (threshold 0.3% improvement)


## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
import traceback
from datetime import datetime
import warnings

from darts import TimeSeries
from darts.models import RandomForestModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor

try:
    from darts.models import SKLearnModel
except ImportError:
    from darts.models import RegressionModel as SKLearnModel

warnings.filterwarnings("ignore")

df_merged = joblib.load("saved_models/df_merged_20260408_1820.joblib")

LEVEL_VARS = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold", "WTI", "GDP"]
RATE_VARS  = ["BI_Rate", "CPI", "NPL_Ratio", "US_Treasury_10Y"]

print(f"Data: {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")


## 2. Konfigurasi Eksperimen

In [ ]:
SINGLE_COVARIATES = {
    "None":            [],
    # Macro (7)
    "BI_Rate":         ["BI_Rate"],
    "CPI":             ["CPI"],
    "M2":              ["M2"],
    "NPL_Ratio":       ["NPL_Ratio"],
    "USDIDR":          ["USDIDR"],
    "GDP":             ["GDP"],
    "US_Treasury_10Y": ["US_Treasury_10Y"],
    # Commodity (7)
    "Coal":            ["Coal"],
    "Copper":          ["Copper"],
    "Nickel":          ["Nickel"],
    "Silver":          ["Silver"],
    "Tin":             ["Tin"],
    "Gold":            ["Gold"],
    "WTI":             ["WTI"],
    # Regional (1)
    "STI":             ["STI"],
}

WINDOWS = [30, 120]
HORIZON = 1
N_FOLDS = 5

# Default hyperparameters (spec-based, bukan arbitrary)
# Phase 2 akan pakai hyperparameters yang di-tune via Optuna untuk best config dari Phase 1
MODEL_CONFIGS = {
    "RandomForest": {
        "n_estimators": 300,
        "max_depth": 5,
        "max_features": "sqrt",
        "max_samples": 0.7,
    },
    "ExtraTrees": {
        "n_estimators": 300,
        "max_depth": 5,
        "max_features": "sqrt",
        # bootstrap=False default -> max_samples diabaikan
    },
    "XGBoost": {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.1,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "reg_alpha": 0.01,
        "reg_lambda": 1.0,
    },
    "LightGBM": {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "reg_alpha": 0.01,
        "reg_lambda": 1.0,
    },
}

total_exp = len(MODEL_CONFIGS) * len(SINGLE_COVARIATES) * len(WINDOWS)
print(f"Models: {list(MODEL_CONFIGS.keys())}")
print(f"Covariates: {len(SINGLE_COVARIATES)} configs")
print(f"Windows: {WINDOWS}")
print(f"Total experiments: {total_exp}")


## 3. Helper Functions

In [ ]:
def to_series(df, target_col, covariates=None):
    """Convert DataFrame to Darts TimeSeries (target + optional covariates)."""
    target = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B",
    )
    target = fill_missing_values(target)

    cov = None
    if covariates:
        cov = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B",
        )
        cov = fill_missing_values(cov)
    return target, cov


def build_model(model_name, window, has_covariates):
    """Factory: build a Darts model with default params from MODEL_CONFIGS."""
    params = MODEL_CONFIGS[model_name]
    common = {
        "lags": window,
        "lags_past_covariates": window if has_covariates else None,
        "output_chunk_length": HORIZON,
    }

    if model_name == "RandomForest":
        return RandomForestModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "ExtraTrees":
        return SKLearnModel(
            **common,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **params),
        )
    elif model_name == "XGBoost":
        return XGBModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "LightGBM":
        return LightGBMModel(**common, random_state=42, n_jobs=-1, verbose=-1, **params)
    else:
        raise ValueError(f"Unknown model: {model_name}")


print("Helpers defined")


## 4. Evaluate Function (5-fold Expanding CV, MAPE on Level)

Pipeline per fold:
1. Split expanding: train[:train_end], test[train_end:test_end]
2. Transform target: `log -> Diff(1) -> Scaler` (fit on train)
3. Transform covariates: `log -> Diff(1)` untuk LEVEL_VARS, `Diff(1)` untuk RATE_VARS, lalu `Scaler`
4. Fit model on train, historical forecast on test window
5. Inverse: scaler -> log-diff -> cumsum dari anchor -> exp -> harga level
6. Compute metrics: MAPE, MAE, RMSE, R2, DA


In [ ]:
def train_and_evaluate(model_name, target_ts, cov_ts, window, n_folds=N_FOLDS):
    """5-fold expanding CV. Returns dict of per-fold metrics."""
    n = len(target_ts)
    test_size = int(n * 0.15)
    min_train = int(n * 0.4)
    available = n - min_train - test_size
    step = max(1, available // max(1, n_folds - 1))

    fold_metrics = {"mape": [], "mae": [], "rmse": [], "r2": [], "da": []}

    for fold in range(n_folds):
        train_end = min_train + fold * step
        test_end = min(train_end + test_size, n)
        if test_end > n or train_end >= test_end:
            break

        train_ts = target_ts[:train_end]
        test_ts = target_ts[train_end:test_end]
        fold_ts = target_ts[:test_end]

        # Target transform: log -> diff -> scale (fit on train)
        train_log = train_ts.map(np.log)
        fold_log = fold_ts.map(np.log)
        differencer = Diff(lags=1)
        train_log_diff = differencer.fit_transform(train_log)
        fold_log_diff = differencer.transform(fold_log)
        scaler = Scaler()
        train_scaled = scaler.fit_transform(train_log_diff)
        fold_scaled = scaler.transform(fold_log_diff)

        # Covariate transform
        cov_transformed = None
        if cov_ts is not None:
            cov_fold = cov_ts[:test_end]
            cov_train = cov_ts[:train_end]
            cov_cols = cov_ts.components.tolist()
            level_cols = [c for c in cov_cols if c in LEVEL_VARS]
            rate_cols = [c for c in cov_cols if c in RATE_VARS]

            parts_train, parts_fold = [], []
            if level_cols:
                d1 = Diff(lags=1)
                parts_train.append(d1.fit_transform(cov_train[level_cols].map(np.log)))
                parts_fold.append(d1.transform(cov_fold[level_cols].map(np.log)))
            if rate_cols:
                d2 = Diff(lags=1)
                parts_train.append(d2.fit_transform(cov_train[rate_cols]))
                parts_fold.append(d2.transform(cov_fold[rate_cols]))

            ct = parts_train[0]
            cf = parts_fold[0]
            for pt, pf in zip(parts_train[1:], parts_fold[1:]):
                ct = ct.stack(pt)
                cf = cf.stack(pf)

            cov_scaler = Scaler()
            cov_scaler.fit(ct)
            cov_transformed = cov_scaler.transform(cf)

        # Fit model
        model = build_model(model_name, window, has_covariates=(cov_ts is not None))
        model.fit(train_scaled, past_covariates=cov_transformed)

        # Historical forecast
        forecast_list = model.historical_forecasts(
            series=fold_scaled,
            past_covariates=cov_transformed,
            start=test_ts.start_time(),
            forecast_horizon=HORIZON,
            stride=HORIZON,
            retrain=False,
            last_points_only=False,
            verbose=False,
        )
        if isinstance(forecast_list, TimeSeries):
            forecast_list = [forecast_list]

        # Inverse transform to level price
        all_dates, all_prices = [], []
        fold_log_full = fold_ts.map(np.log)
        for chunk_scaled in forecast_list:
            chunk_diff = scaler.inverse_transform(chunk_scaled)
            dates = chunk_diff.time_index
            vals = chunk_diff.values().flatten()
            idx = fold_ts.get_index_at_point(dates[0])
            if idx == 0:
                continue
            anchor = fold_log_full[idx - 1].values()[0][0]
            log_prices = anchor + np.cumsum(vals)
            prices = np.exp(log_prices)
            all_dates.extend(dates)
            all_prices.extend(prices)

        # Metrics
        pred_df = pd.DataFrame({"date": pd.to_datetime(all_dates), "predicted": all_prices})
        actual_df = fold_ts.to_dataframe().reset_index()
        actual_df.columns = ["date", "actual"]
        eval_df = pd.merge(actual_df, pred_df, on="date", how="inner")
        y_true = eval_df["actual"].values
        y_pred = eval_df["predicted"].values

        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        actual_dir = np.diff(y_true)
        pred_dir = y_pred[1:] - y_true[:-1]
        da = np.mean((actual_dir > 0) == (pred_dir > 0)) * 100 if len(actual_dir) > 0 else np.nan

        fold_metrics["mape"].append(mape)
        fold_metrics["mae"].append(mae)
        fold_metrics["rmse"].append(rmse)
        fold_metrics["r2"].append(r2)
        fold_metrics["da"].append(da)

        del model
        gc.collect()

    return fold_metrics


print("Evaluation function defined")


## 5. Run Experiment Loop (128 experiments)

In [ ]:
results = []
start_time = datetime.now()
exp_num = 0
print(f"Phase 1 Screening started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total experiments: {total_exp}")
print("=" * 70)

for model_name in MODEL_CONFIGS.keys():
    for cov_name, cov_vars in SINGLE_COVARIATES.items():
        for window in WINDOWS:
            exp_num += 1
            tag = f"[{exp_num}/{total_exp}] {model_name:13s} | {cov_name:16s} | W{window}"
            try:
                target_ts, cov_ts = to_series(
                    df_merged, "IHSG", cov_vars if cov_vars else None
                )
                fm = train_and_evaluate(model_name, target_ts, cov_ts, window)

                row = {
                    "Model": model_name,
                    "Covariates": cov_name,
                    "Window": window,
                    "MAPE_mean": round(np.mean(fm["mape"]), 4),
                    "MAPE_std":  round(np.std(fm["mape"]), 4),
                    "MAE_mean":  round(np.mean(fm["mae"]), 4),
                    "RMSE_mean": round(np.mean(fm["rmse"]), 4),
                    "RMSE_std":  round(np.std(fm["rmse"]), 4),
                    "R2_mean":   round(np.mean(fm["r2"]), 4),
                    "DA_mean":   round(np.mean(fm["da"]), 2),
                    "DA_std":    round(np.std(fm["da"]), 2),
                }
                results.append(row)
                print(f"{tag} -> MAPE={row['MAPE_mean']:.4f}+-{row['MAPE_std']:.4f}% | DA={row['DA_mean']:.1f}%")
            except Exception as e:
                print(f"{tag} -> ERROR: {e}")
                traceback.print_exc()
                results.append({
                    "Model": model_name, "Covariates": cov_name, "Window": window,
                    "Error": str(e),
                })

elapsed = datetime.now() - start_time
print("=" * 70)
print(f"Done in {elapsed}")

df_results = pd.DataFrame(results)
df_results.to_csv("phase1_screening_results.csv", index=False)
print(f"Saved: phase1_screening_results.csv ({len(df_results)} rows)")
display(df_results.sort_values("MAPE_mean").head(20))


## 6. Screening Analysis

Covariate dianggap "lolos" kalau improvement MAPE terhadap baseline (`None`) >= threshold. Default threshold = **0.3%** (relative improvement).


In [ ]:
def analyze_screening(df, threshold_pct=0.3):
    rows = []
    for model_name in df["Model"].unique():
        for window in df["Window"].unique():
            mask = (df["Model"] == model_name) & (df["Window"] == window)
            subset = df[mask]
            baseline_row = subset[subset["Covariates"] == "None"]
            if baseline_row.empty or "MAPE_mean" not in baseline_row.columns:
                continue
            baseline_mape = baseline_row["MAPE_mean"].values[0]

            for _, row in subset.iterrows():
                if row["Covariates"] == "None":
                    continue
                if pd.isna(row.get("MAPE_mean")):
                    continue
                impr = (baseline_mape - row["MAPE_mean"]) / baseline_mape * 100
                rows.append({
                    "Model": model_name,
                    "Window": window,
                    "Covariate": row["Covariates"],
                    "MAPE": row["MAPE_mean"],
                    "Baseline_MAPE": baseline_mape,
                    "Improvement_pct": round(impr, 3),
                    "Passed": impr >= threshold_pct,
                })
    return pd.DataFrame(rows)


df_screening = analyze_screening(df_results, threshold_pct=0.3)
df_screening.to_csv("phase1_screening_analysis.csv", index=False)
print(f"Saved: phase1_screening_analysis.csv")

# Pass counts (out of 4 models x 2 windows = 8)
pass_counts = df_screening[df_screening["Passed"]].groupby("Covariate").size().sort_values(ascending=False)
print("\nCovariate pass counts (out of 8 model x window combos):")
display(pass_counts.to_frame("pass_count"))
